In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.stats import (
    shapiro,
    pearsonr,
    spearmanr,
    mannwhitneyu,
    ttest_ind,
    ttest_rel,
    wilcoxon,
    chi2_contingency
)
MAIN_FOLDER = "../"
PELLG_FOLDER = "5-right_left_pellgregory_agreement-2/"

In [ ]:
# ============================================================
# 2. Load dataset
# ============================================================
FILE_NAME = "paired_list"

# Read the excel file
paired = pd.read_excel("../../"+FILE_NAME+".xlsx")

# Show the head
print(paired.tail())

In [ ]:
# Standardize column names
paired.columns = paired.columns.str.strip().str.lower()

# Create output folder
output_folder = MAIN_FOLDER+PELLG_FOLDER
os.makedirs(output_folder, exist_ok=True)

# Clean Pell-Gregory columns
paired["pell_gregory_left"] = (
    paired["pell_gregory_left"]
    .astype(str)
    .str.strip()
    .str.upper()
)

paired["pell_gregory_right"] = (
    paired["pell_gregory_right"]
    .astype(str)
    .str.strip()
    .str.upper()
)

# Replace text-like missing values with NaN
missing_values = ["", "NAN", "NONE", "NULL", "YOK"]

paired["pell_gregory_left"] = paired["pell_gregory_left"].replace(missing_values, np.nan)
paired["pell_gregory_right"] = paired["pell_gregory_right"].replace(missing_values, np.nan)

# Keep only patients with both left and right Pell-Gregory values
pell_df = paired.dropna(subset=["pell_gregory_left", "pell_gregory_right"]).copy()

# Check exact left-right Pell-Gregory match
pell_df["pell_gregory_exact_match"] = (
    pell_df["pell_gregory_left"] == pell_df["pell_gregory_right"]
)

In [ ]:
pell_df.tail()

In [ ]:
# Calculate match statistics
total_patients = len(pell_df)
matched_patients = pell_df["pell_gregory_exact_match"].sum()
unmatched_patients = total_patients - matched_patients

match_percentage = matched_patients / total_patients * 100
unmatch_percentage = unmatched_patients / total_patients * 100

summary = pd.DataFrame({
    "Metric": [
        "Total patients with bilateral Pell-Gregory data",
        "Exact matched patients",
        "Unmatched patients",
        "Exact match percentage",
        "Unmatched percentage"
    ],
    "Value": [
        total_patients,
        matched_patients,
        unmatched_patients,
        round(match_percentage, 2),
        round(unmatch_percentage, 2)
    ]
})

summary

In [ ]:
match_counts = pell_df["pell_gregory_exact_match"].value_counts()

match_counts.index = match_counts.index.map({
    True: "Exact Match",
    False: "No Exact Match"
})

match_table = pd.DataFrame({
    "Count": match_counts,
    "Percentage (%)": (match_counts / match_counts.sum() * 100).round(2)
})

print(match_table)

match_table.to_excel(
    os.path.join(output_folder, "pell_gregory_exact_match_summary.xlsx")
)

In [ ]:
pell_crosstab = pd.crosstab(
    pell_df["pell_gregory_left"],
    pell_df["pell_gregory_right"]
)

pell_crosstab_percent = pd.crosstab(
    pell_df["pell_gregory_left"],
    pell_df["pell_gregory_right"],
    normalize="index"
) * 100

print("Left vs Right Pell-Gregory Crosstab")
print(pell_crosstab)

print("Left vs Right Pell-Gregory Crosstab (%)")
print(pell_crosstab_percent.round(2))

pell_crosstab.to_excel(
    os.path.join(output_folder, "left_right_pell_gregory_crosstab.xlsx")
)

pell_crosstab_percent.round(2).to_excel(
    os.path.join(output_folder, "left_right_pell_gregory_crosstab_percent.xlsx")
)

In [ ]:
from sklearn.metrics import cohen_kappa_score

kappa = cohen_kappa_score(
    pell_df["pell_gregory_left"],
    pell_df["pell_gregory_right"]
)

kappa_result = pd.DataFrame({
    "Metric": ["Cohen's Kappa"],
    "Value": [round(kappa, 4)]
})

print(kappa_result)

kappa_result.to_excel(
    os.path.join(output_folder, "pell_gregory_cohens_kappa.xlsx"),
    index=False
)

In [ ]:
def my_autopct(pct):
    return ("%1.1f%%" % pct) if pct > 1.0 else ""

# ---------------------------
# Global style settings for academic appearance
# ---------------------------
plt.rcParams.update({
    "font.family": "serif",
    "font.serif": ["Times New Roman", "DejaVu Serif", "serif"],
    "font.size": 12,
    "axes.titlesize": 14,
    "axes.labelsize": 12,
    "xtick.labelsize": 11,
    "ytick.labelsize": 11,
    "legend.fontsize": 11,
    "figure.dpi": 300,
    "savefig.dpi": 300
})

plt.figure(figsize=(8, 8))

colors = plt.cm.Blues(np.linspace(0.45, 0.85, len(match_counts)))

wedges, texts, autotexts = plt.pie(
    match_counts,
    autopct=my_autopct,
    startangle=90,
    pctdistance=0.85,
    colors=colors,
    wedgeprops=dict(width=0.5, edgecolor="w")
)

# Set percentage text style
for autotext in autotexts:
    autotext.set_color("white")
    autotext.set_fontsize(12)
    autotext.set_fontweight("bold")

plt.title("Exact Match Rate of Left and Right Pell-Gregory Classes")


plt.legend(
    wedges,
    match_counts.index.astype(str),
    title="Match Status",
    loc="center left",
    bbox_to_anchor=(1, 0, 0.5, 1)
)

plt.tight_layout()

plt.savefig(
    os.path.join(output_folder, "pell_gregory_exact_match_donut_chart.png"),
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
import matplotlib.colors as mcolors

# ---------------------------
# Global style settings for academic appearance
# ---------------------------
plt.rcParams.update({
    "font.family": "serif",
    "font.serif": ["Times New Roman", "DejaVu Serif", "serif"],
    "font.size": 15,
    "axes.titlesize": 15,
    "axes.labelsize": 15,
    "xtick.labelsize": 15,
    "ytick.labelsize": 15,
    "legend.fontsize": 15,
    "figure.dpi": 300,
    "savefig.dpi": 300
})

plt.figure(figsize=(8, 6))

#img = plt.imshow(pell_crosstab, aspect="auto", cmap="magma")
# ---------------------------
# Create a darker blue colormap
# ---------------------------
base_cmap = plt.cm.Blues

# Use only the medium-to-dark part of the Blues colormap
# 0.25 avoids very pale blue/white colors
# 0.90 avoids extremely dark colors
blue_colors = base_cmap(np.linspace(0.15, 0.95, 256))

custom_blues = mcolors.LinearSegmentedColormap.from_list(
    "custom_blues",
    blue_colors
)

img = plt.imshow(pell_crosstab, aspect="auto", cmap=custom_blues)

plt.title("(b) Pell-Gregory class")
plt.xlabel("Right side")
plt.ylabel("Left side")

plt.xticks(
    ticks=np.arange(len(pell_crosstab.columns)),
    labels=pell_crosstab.columns,
    rotation=0,
    ha="center"
)

plt.yticks(
    ticks=np.arange(len(pell_crosstab.index)),
    labels=pell_crosstab.index
)

# Threshold controls when text becomes white.
threshold = pell_crosstab.values.max() * 0.45

for i in range(len(pell_crosstab.index)):
    for j in range(len(pell_crosstab.columns)):
        value = pell_crosstab.iloc[i, j]

        # Use white text on darker cells, dark text on lighter cells
        text_color = "white" if value >= threshold else "#222222"
        
        plt.text(
            j,
            i,
            str(value),
            ha="center",
            va="center",
            color=text_color,
            fontsize=15,
            fontweight="bold"
        )

plt.colorbar(img, label="Number of Patients")
plt.tight_layout()

plt.savefig(
    os.path.join(output_folder, "left_right_pell_gregory_crosstab_heatmap.png"),
    dpi=300,
    bbox_inches="tight"
)

plt.show()